# Phase 2 — Google Colab Execution Notebook

This notebook runs the tiered data pipeline in Google Colab.
- Working directory: Project root (`/content/drive/MyDrive/Ultra-Dataa` or `/content/Ultra-Dataa`)
- Pipeline stages: L0 Substitute Generation → L1 Filtering → L2 Selection

In [ ]:
# Cell 1: Mount Google Drive or clone the repository
# Option A: Mount Google Drive (standard workflow if synced with Google Drive)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Note: Drive mount skipped or running in non-Colab environment: {e}")

# Option B: Git Clone (uncomment if pulling directly from GitHub/GitLab instead of Drive)
# !git clone https://github.com/<your-username>/Ultra-Dataa.git /content/Ultra-Dataa

In [ ]:
# Cell 2: Change directory into the project root
import os

# Set path to project root (adjust if your Drive directory has a different name)
drive_project_path = "/content/drive/MyDrive/Ultra-Dataa"
cloned_project_path = "/content/Ultra-Dataa"

if os.path.exists(drive_project_path):
    os.chdir(drive_project_path)
elif os.path.exists(cloned_project_path):
    os.chdir(cloned_project_path)
else:
    print(f"Warning: Neither {drive_project_path} nor {cloned_project_path} found. Staying in current directory: {os.getcwd()}")

!pwd
!ls -la

In [ ]:
# Cell 3: Install required packages if missing
!pip install -q -r requirements.txt

In [ ]:
# Cell 4: Set PYTHONPATH
import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.environ["PYTHONPATH"] = "."
print(f"Project root (CWD): {project_root}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH')}")

In [ ]:
# Cell 5: Generate expanded L0 substitute data
!PYTHONPATH=. python scripts/generate_l0_expanded.py

In [ ]:
# Cell 6: Run L1 expanded pipeline
!PYTHONPATH=. python scripts/run_l1.py --config configs/l1_expanded.yaml

In [ ]:
# Cell 7: Run L2 pipeline
!PYTHONPATH=. python scripts/run_l2.py --config configs/l2_expanded.yaml

In [ ]:
# Cell 8: Print output file locations
from pathlib import Path

print("=" * 60)
print("Pipeline Output File Locations & Status")
print("=" * 60)

data_dirs = [
    Path("data/l0_raw"),
    Path("data/l1_filtered"),
    Path("data/l2_selected"),
]

for directory in data_dirs:
    posix_path = directory.as_posix()
    print(f"\n[Directory] {posix_path}/")
    if directory.exists():
        files = sorted([f for f in directory.iterdir() if f.is_file()])
        if files:
            for f in files:
                size_kb = f.stat().st_size / 1024
                print(f"  -> {f.as_posix()} ({size_kb:.2f} KB)")
        else:
            print("  -> (empty)")
    else:
        print("  -> (directory does not exist yet)")

print("\n" + "=" * 60)